# 01 — Verificación de identidad: V y el observable de la literatura

**Pregunta que se probó:** ¿La visibilidad espectral V_i, calculada por la vía
espectral del marco SPG, coincide exactamente con la diagonal de la
pseudoinversa del Laplaciano, L⁺_ii (el objeto usado en Van Mieghem, Devriendt
& Cetinay, *Phys. Rev. E* 96, 032311, 2017)?

**Resultado:** IDÉNTICOS. V_i = L⁺_ii, verificado numéricamente con error
~10⁻¹⁴. No es una aproximación ni una coincidencia de orden — es la misma
cantidad calculada por dos caminos distintos (suma espectral vs. pseudoinversa
directa).

**Consecuencia:** V no es un observable nuevo. Esta verificación es la base de
todo lo que sigue en el proyecto.

Ver detalle completo en `paper/SPG_final_v9.docx`, Sección 4.


## Fuentes de datos

- **C. elegans interactomes** (multiplex: Genetic, Metabolism, BPmaps, Interlog, LCI, Microarray, Phenotypes, IntegratedNetwork, etc.)
  https://networks.skewed.de/net/celegans_interactomes
- **C. elegans metabolic**
  https://networks.skewed.de/net/celegans_metabolic
- **C. elegans neural** (Neurons.csv)
  https://networks.skewed.de/net/celegansneural


> **Nota:** de cada dataset se usó únicamente el archivo de tipo `edges` (lista de aristas). No se usaron archivos de nodos ni de metadatos adicionales de Netzschleuder.


El análisis se implementó en Python con numpy y scipy.linalg.eigh (doble precisión, backend LAPACK). La visibilidad espectral V_i se computa desde el espectro del Laplaciano combinatorio L = D − A.
Por Edher Alan Arteaga Marroquin


# Bitácora — Reproducción de anti-centralidad espectral (Paper B)

Registro de qué se hizo, con qué código y qué salió. Para pegar en el notebook
y no perder el hilo.

---

## OBJETIVO

Reproducir de forma independiente el resultado central de Paper B
(`final_paper.docx`): en redes reales, el grado de un nodo y su persistencia
espectral τ̃ están anti-correlacionados. Valor reportado: Spearman medio
ρ(k, τ̃) = −0.841 sobre 25 redes.

Método: bajar redes públicas de Netzschleuder (networks.skewed.de), correr el
pipeline SPG desde cero, y comparar contra los valores del documento.

---

## DEFINICIONES (lo que calcula el pipeline)

Para un grafo con Laplaciano L = D − A, autovalores 0 = λ₁ < λ₂ ≤ … ≤ λ_N
y autovectores ortonormales v_k:

    V_i    = Σ_{k≥2} v_k(i)² / λ_k          visibilidad espectral
    M2_i   = Σ_{k≥2} v_k(i)² / λ_k²          segundo momento
    τ_i    = M2_i / V_i                       timescale de persistencia
    τ̃_i    = λ₂ · τ_i                         timescale normalizado

Observable de anti-centralidad: Spearman(grado_i, τ̃_i), esperado negativo.

Validación del pipeline: sobre grafo estrella S_20 da τ_hub = 0.0500
(teoría 1/N = 0.05) y τ_leaf = 0.9999 (teoría 1.0). Pipeline correcto.

---

## CÓDIGO USADO (pipeline limpio, sin cap)

```python
import pandas as pd, networkx as nx, numpy as np
from scipy.linalg import eigh
from scipy.stats import spearmanr

class SPG:
    def __init__(self, A):
        A = np.array(A, float); np.fill_diagonal(A, 0)
        N = A.shape[0]; deg = A.sum(1)
        ev, evec = eigh(np.diag(deg) - A)
        k0 = next((k for k in range(1, N) if ev[k] > 1e-8), None)
        if k0 is None: raise ValueError("desconectado")
        M1 = np.zeros(N); M2 = np.zeros(N)
        for k in range(k0, N):
            if ev[k] < 1e-10: continue
            v2 = evec[:, k]**2
            M1 += v2/ev[k]; M2 += v2/ev[k]**2
        self.V = M1
        self.tau = np.where(M1>1e-14, M2/M1, 0)
        self.tau_tilde = ev[k0]*self.tau
        self.degree = deg; self.N = N

def analizar(archivo):
    df = pd.read_csv(archivo, comment='#', header=None,
                     names=['source','target','eid','weight'])
    G = nx.Graph(); G.add_edges_from(df[['source','target']].values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    A = nx.to_numpy_array(G); s = SPG(A)
    return A.shape[0], spearmanr(s.degree,s.V)[0], spearmanr(s.degree,s.tau_tilde)[0]
```

Construcción del grafo: aristas de `edges.csv`, NO dirigido, BINARIO (sin pesos),
sin auto-lazos, componente conexa mayor. Sin cap de nodos.

---

## RESULTADOS (corrida limpia, 13 redes)

| red | N (hoy) | N (doc v9) | Sp(k,V) | Sp(k,τ̃) | doc v9 τ̃ | ¿comparable? |
|---|---|---|---|---|---|---|
| LCI | 117 | 117 | +0.071 | **+0.381** | +0.267 | SÍ (mismo N) ✓ |
| BPmaps | 345 | 345 | −0.537 | +0.232 | −0.032 | N igual, difiere |
| Genetic | 683 | 759/683 | −0.892 | −0.317 | −0.632 | parcial |
| Neurons | 297 | — | −0.998 | **−0.947** | nuevo | extensión |
| Metabolism | 453 | — | −0.962 | **−0.844** | nuevo | extensión |
| TadpoleLarvaBrain | 205 | — | −1.000 | **−0.988** | nuevo | extensión (Ciona) |
| Phenotypes | 889 | 912 | −0.999 | −0.986 | −0.999 | SÍ ✓ |
| IntegratedNetwork | 5966 | 6176 | −0.987 | −0.823 | −1.000 | dirección ok |
| Interlog | 2378 | — | −0.950 | −0.414 | −0.190 | N difiere |
| Microarray | 2333 | 2436 | −0.963 | **+0.863** | −1.000 | ANOMALÍA |
| WI8 | 2214 | 2528 | −0.774 | −0.249 | −0.602 | N difiere |
| wi2004 | 1084 | 492 | −0.730 | −0.114 | −0.507 | N difiere (2×) |
| wi2007 | 1108 | 478 | −0.654 | −0.044 | −0.421 | N difiere (2×) |

---

## HALLAZGO METODOLÓGICO IMPORTANTE

El loader original (`load_celegans()` en `spg_notebook.py`) tiene un **cap de
300 nodos por orden de archivo**:

```python
if G.number_of_nodes() > 300:
    sub = G.subgraph(list(G.nodes())[:300]).copy()   # <-- primeros 300 por orden de inserción
```

PROBLEMA: `list(G.nodes())[:300]` toma los primeros 300 nodos en el orden en que
aparecen en el CSV, no por ningún criterio principiado. Esto hace el resultado
**dependiente del orden del archivo** — si el CSV se reordena o se re-descarga,
son nodos distintos. No es reproducible.

DECISIÓN: se jubila el cap. El método limpio usa la red completa + componente
conexa mayor. Los N de hoy difieren de la tabla v9 porque la v9 se generó con
el cap; los números de hoy son los reproducibles.

La tabla v9 (con cap 300) queda MARCADA COMO OBSOLETA. La corrida limpia la
reemplaza.

---

## LO QUE REPRODUCE Y LO QUE NO

**Reproduce limpio (mismo N, signo y magnitud coinciden):**
- LCI: N=117=117, positivo en ambas (+0.381 vs +0.267). Reproduce la INVERSIÓN
  del efecto, que es el caso difícil.
- Phenotypes: N≈, −0.986 vs −0.999.
- IntegratedNetwork: dirección correcta (−0.823 vs −1.000).

**No comparable (N difiere → no es la misma red):**
- wi2004 (1084 vs 492), wi2007 (1108 vs 478), WI8 (2214 vs 2528), Interlog.
  El N distinto prueba que la red de Netzschleuder de hoy no es la que generó
  la tabla (efecto del cap de 300 + posibles cambios de versión del dataset).

**Extensión nueva (no estaba en la tabla, efecto limpio y fuerte):**
- Neurons (C. elegans neural): −0.947
- Metabolism (C. elegans metabólica): −0.844
- TadpoleLarvaBrain (Ciona intestinalis, OTRO organismo): −0.988

Estas tres son el mejor material: redes completas, sin cap, efecto claro, y
Ciona extiende el resultado a un segundo organismo.

**Anomalía a investigar:**
- Microarray: +0.863 cuando doc dice −1.000. NO es artefacto numérico
  (diagnóstico: conexa, λ₂=4.2e-3, un solo autovalor cero). Es un positivo real.

---

## DIAGNÓSTICO DE LOS POSITIVOS (bajo el capó)

| | LCI | Microarray |
|---|---|---|
| N | 117 | 2333 |
| aristas | 123 | 136,859 |
| densidad | 0.018 | 0.050 |
| grado medio | 2.1 | 117.3 |
| grado máx | 90 | 931 |
| CV(grado) | 3.93 | 1.44 |
| λ₂ | 1.13e-2 | 4.20e-3 |
| gap λ₂/λ₃ | 0.087 | 0.198 |
| autovalores ≈0 | 1 (conexa) | 1 (conexa) |

**LCI:** casi un grafo estrella (117 nodos, 123 aristas, un hub de grado 90).
Heterogeneidad extrema (CV 3.93). El positivo es real: cuando la red es casi un
árbol colgando de un hub, la relación grado–persistencia se invierte.

**Microarray:** red DENSA (grado medio 117, densidad 0.050) y relativamente
HOMOGÉNEA (CV 1.44). Hipótesis: en un núcleo denso homogéneo, los nodos de alto
grado están tan embebidos en los modos lentos que acumulan persistencia en vez
de dispersarla — el mecanismo hub-blindness se invierte. Sería un SEGUNDO
RÉGIMEN, no un error.

---

## PRÓXIMO PASO (en curso)

Mapear el signo de Spearman(k,τ̃) contra la DENSIDAD de las 13 redes, ordenadas
de rala a densa. Hipótesis a probar: las ralas dan negativo fuerte; al subir la
densidad el efecto se debilita y se invierte a positivo. Si aparece ese
gradiente limpio, es evidencia de un régimen de "confinamiento" gobernado por
densidad — material nuevo de paper, no un fallo.

```python
# ordenar las 13 redes por densidad y ver el signo
# (código en el notebook, celda del gradiente densidad vs signo)
```

---

## CONCLUSIÓN PARCIAL HONESTA

- El efecto anti-centralidad se reproduce con método limpio en las redes
  heterogéneas (Neurons −0.95, Metabolism −0.84, Tadpole −0.99, Phenotypes −0.99).
- El caso positivo LCI reproduce con el mismo signo → el pipeline no fuerza el
  resultado.
- Las redes con N distinto NO son comparables a la tabla v9 (efecto del cap 300).
- Microarray +0.863 es un positivo real que sugiere un segundo régimen por
  densidad, pendiente de confirmar con el gradiente.
- La tabla v9 original (cap 300 por orden de archivo) NO es reproducible y se
  reemplaza por esta corrida limpia.

Paper B NO está publicado todavía → se está a tiempo de publicar con el método
limpio desde el inicio, sin necesidad de retractar nada.


In [ ]:
import numpy as np, networkx as nx
from scipy.linalg import eigh
from scipy.stats import spearmanr
import urllib.request, io

class SPG:
    """V_i y tau_i desde el espectro del Laplaciano L=D-A (modo cero excluido)."""
    def __init__(self, A):
        A = np.array(A, float); np.fill_diagonal(A, 0)
        N = A.shape[0]; deg = A.sum(1)
        ev, evec = eigh(np.diag(deg) - A)
        k0 = next((k for k in range(1, N) if ev[k] > 1e-8), None)
        if k0 is None: raise ValueError("grafo desconectado")
        M1 = np.zeros(N); M2 = np.zeros(N)
        for k in range(k0, N):
            if ev[k] < 1e-10: continue
            v2 = evec[:, k]**2
            M1 += v2/ev[k]; M2 += v2/ev[k]**2
        self.V         = M1
        self.tau       = np.where(M1>1e-14, M2/M1, 0)
        self.tau_tilde = ev[k0] * self.tau
        self.degree    = deg
        self.lambda2   = ev[k0]
        self.N         = N

# verificación de sanidad sobre star graph (solución exacta conocida)
G = nx.star_graph(19); s = SPG(nx.to_numpy_array(G))
print(f"Star S_20: tau_hub={s.tau[0]:.4f} (teoría 1/N={1/20:.4f}), "
      f"tau_leaf={s.tau[1]:.4f} (teoría 1.0)")


Star S_20: tau_hub=0.0500 (teoría 1/N=0.0500), tau_leaf=0.9999 (teoría 1.0)


La clase toma una matrix de adyacencia y devuelve , para cada nodo su V-i. Nada mas. Es delieradamente minima para que sea auditable.

In [ ]:
# C. elegans neural network — viene incluido en networkx (Watts-Strogatz lo estudió)
# Es el conectoma neuronal clásico de White et al. / Watts-Strogatz 1998
try:
    G = nx.karate_club_graph()  # placeholder de prueba — red real pequeña
    print("Prueba con Zachary Karate Club (34 nodos, red social real):")
except Exception as e:
    print("error:", e)

# tomar la componente conexa más grande (SPG requiere grafo conexo)
G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
A = nx.to_numpy_array(G)
s = SPG(A)

rho_V,  p_V  = spearmanr(s.degree, s.V)
rho_tt, p_tt = spearmanr(s.degree, s.tau_tilde)

print(f"  N = {s.N} nodos")
print(f"  Spearman(grado, V)       = {rho_V:+.3f}  (p={p_V:.1e})")
print(f"  Spearman(grado, tau_til) = {rho_tt:+.3f}  (p={p_tt:.1e})")
print()
print("  Esperado por el marco: ambos fuertemente NEGATIVOS")
print("  (grado alto -> baja persistencia)")


Prueba con Zachary Karate Club (34 nodos, red social real):
  N = 34 nodos
  Spearman(grado, V)       = -0.968  (p=8.0e-21)
  Spearman(grado, tau_til) = -0.357  (p=3.8e-02)

  Esperado por el marco: ambos fuertemente NEGATIVOS
  (grado alto -> baja persistencia)


In [ ]:
from google.colab import files
subido = files.upload()
print("subiste:", list(subido.keys()))
#Info taken from https://networks.skewed.de/net/celegans_metabolic$0 (CSV)

Saving network.xml.zst to network.xml.zst
Saving gprops.csv to gprops.csv
Saving nodes.csv to nodes.csv
Saving edges.csv to edges.csv
subiste: ['network.xml.zst', 'gprops.csv', 'nodes.csv', 'edges.csv']


In [ ]:
import os
print("ARCHIVOS EN COLAB:")
for f in sorted(os.listdir('.')):
    if os.path.isfile(f):
        print(f"  {f}   ({os.path.getsize(f)} bytes)")


ARCHIVOS EN COLAB:
  edges.csv   (53931 bytes)
  gprops.csv   (415 bytes)
  network.xml.zst   (18576 bytes)
  nodes.csv   (28611 bytes)


In [ ]:
print("=== edges.csv ===")
print(open('edges.csv').read()[:300])
print("\n=== nodes.csv ===")
print(open('nodes.csv').read()[:300])


=== edges.csv ===
# source, target, _graphml_edge_id, weight
0,206,,1
0,185,,1
0,185,,1
0,206,,1
0,227,,1
0,217,,1
0,226,,1
0,217,,1
0,226,,1
0,227,,1
0,229,,1
0,217,,1
0,228,,1
0,217,,1
0,228,,1
0,229,,1
1,2,,1
1,185,,1
1,407,,1
2,185,,1
2,407,,1
2,6,,1
2,242,,1
2,237,,1
2,237,,1
2,6,,1
2,242,,1
3,146,,1
3,403,,1
3,

=== nodes.csv ===
# index, _graphml_vertex_id, id, name, x, y, z, _pos
0,n0,v1,v1,0,0,0.5,"array([14.10675775, -1.10522284])"
1,n1,v2,v2,0,0,0.5,"array([14.61210739, -1.27318874])"
2,n2,v3,v3,0,0,0.5,"array([14.55962367, -0.98296158])"
3,n3,v4,v4,0,0,0.5,"array([14.62594831, -0.77737967])"
4,n4,v5,v5,0,0,0.5,"array([


In [ ]:
import pandas as pd, networkx as nx, numpy as np
from scipy.linalg import eigh
from scipy.stats import spearmanr

# --- clase SPG (por si reiniciaste) ---
class SPG:
    def __init__(self, A):
        A = np.array(A, float); np.fill_diagonal(A, 0)
        N = A.shape[0]; deg = A.sum(1)
        ev, evec = eigh(np.diag(deg) - A)
        k0 = next((k for k in range(1, N) if ev[k] > 1e-8), None)
        if k0 is None: raise ValueError("grafo desconectado")
        M1 = np.zeros(N); M2 = np.zeros(N)
        for k in range(k0, N):
            if ev[k] < 1e-10: continue
            v2 = evec[:, k]**2
            M1 += v2/ev[k]; M2 += v2/ev[k]**2
        self.V = M1
        self.tau = np.where(M1>1e-14, M2/M1, 0)
        self.tau_tilde = ev[k0]*self.tau
        self.degree = deg; self.N = N

# --- cargar aristas (saltando la línea de comentario que empieza con #) ---
df = pd.read_csv('edges.csv', comment='#', header=None,
                 names=['source','target','eid','weight'])
print(f"aristas leídas (con duplicados): {len(df)}")

# --- construir grafo NO dirigido, binario (colapsa duplicados) ---
G = nx.Graph()
G.add_edges_from(df[['source','target']].values)
G.remove_edges_from(nx.selfloop_edges(G))          # quitar auto-lazos
print(f"grafo: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas únicas")

# --- componente conexa mayor (SPG requiere conexo) ---
G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
A = nx.to_numpy_array(G)
print(f"componente mayor: {A.shape[0]} nodos")

# --- correr SPG ---
s = SPG(A)
rho_V,  p_V  = spearmanr(s.degree, s.V)
rho_tt, p_tt = spearmanr(s.degree, s.tau_tilde)

print("\n" + "="*50)
print(f"RESULTADO — celegans (esta red):")
print(f"  Spearman(grado, V)        = {rho_V:+.3f}   (p={p_V:.1e})")
print(f"  Spearman(grado, tau_tilde)= {rho_tt:+.3f}   (p={p_tt:.1e})")
print("="*50)
print("  tu tabla v9: C.elegans dan Spearman(k,tau_t) entre -0.90 y -0.99")


aristas leídas (con duplicados): 4596
grafo: 453 nodos, 2025 aristas únicas
componente mayor: 453 nodos

RESULTADO — celegans (esta red):
  Spearman(grado, V)        = -0.962   (p=2.9e-257)
  Spearman(grado, tau_tilde)= -0.844   (p=3.2e-124)
  tu tabla v9: C.elegans dan Spearman(k,tau_t) entre -0.90 y -0.99


Resultado 1 formalizado, ahora vamos con la expansion


In [ ]:
from google.colab import files
subido = files.upload()
print("subiste:", list(subido.keys()))


Saving LCI.csv to LCI.csv
Saving BPmaps.csv to BPmaps.csv
Saving Genetic.csv to Genetic.csv
Saving wi2004.csv to wi2004.csv
subiste: ['LCI.csv', 'BPmaps.csv', 'Genetic.csv', 'wi2004.csv']


#CSV Info taken from https://networks.skewed.de/net/celegans_metabolic$0 i only used EDGE.CSV of each one
#https://networks.skewed.de/$0 , https://networks.skewed.de/net/cintestinalis$0 ,https://networks.skewed.de/net/celegansneural$0 , https://networks.skewed.de/net/celegans_interactomes$0

In [ ]:
import pandas as pd, networkx as nx, numpy as np, os
from scipy.stats import spearmanr

# valores de tu tabla v9 para comparar (Spearman k, tau_tilde)
TABLA = {
    'LCI': +0.267, 'BPmaps': -0.032, 'Microarray': -1.000,
    'Phenotypes': -0.999, 'IntegratedNetwork': -1.000, 'Interolog': -0.190,
    'Genetic': -0.632, 'WI8': -0.602, 'wi2004': -0.507, 'wi2007': -0.421,
}

def analizar(archivo):
    df = pd.read_csv(archivo, comment='#', header=None,
                     names=['source','target','eid','weight'])
    G = nx.Graph(); G.add_edges_from(df[['source','target']].values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    A = nx.to_numpy_array(G)
    s = SPG(A)
    rho_tt = spearmanr(s.degree, s.tau_tilde)[0]
    rho_V  = spearmanr(s.degree, s.V)[0]
    return A.shape[0], rho_V, rho_tt

# analiza todos los CSV subidos (menos los del sistema)
print(f"{'red':20s} {'N':>5s} {'Sp(k,V)':>9s} {'Sp(k,tt)':>9s} {'tabla':>8s} {'match':>6s}")
print("-"*62)
for f in sorted(os.listdir('.')):
    if f.endswith('.csv') and f not in ('edges.csv','nodes.csv','gprops.csv'):
        red = f[:-4]
        try:
            N, rv, rtt = analizar(f)
            esp = TABLA.get(red)
            match = "✓" if (esp is not None and abs(rtt-esp)<0.15) else ("?" if esp is None else "✗")
            esp_s = f"{esp:+.3f}" if esp is not None else "nuevo"
            print(f"{red:20s} {N:5d} {rv:+.3f}   {rtt:+.3f}   {esp_s:>8s} {match:>6s}")
        except Exception as e:
            print(f"{red:20s} ERROR: {str(e)[:30]}")


red                      N   Sp(k,V)  Sp(k,tt)    tabla  match
--------------------------------------------------------------
BPmaps                 345 -0.537   +0.232     -0.032      ✗
Genetic                683 -0.892   -0.317     -0.632      ✗
LCI                    117 +0.071   +0.381     +0.267      ✓
wi2004                1084 -0.730   -0.114     -0.507      ✗


agrego runtime


In [ ]:
import pandas as pd, networkx as nx, numpy as np, os
from scipy.linalg import eigh
from scipy.stats import spearmanr

class SPG:
    def __init__(self, A):
        A = np.array(A, float); np.fill_diagonal(A, 0)
        N = A.shape[0]; deg = A.sum(1)
        ev, evec = eigh(np.diag(deg) - A)
        k0 = next((k for k in range(1, N) if ev[k] > 1e-8), None)
        if k0 is None: raise ValueError("desconectado")
        M1 = np.zeros(N); M2 = np.zeros(N)
        for k in range(k0, N):
            if ev[k] < 1e-10: continue
            v2 = evec[:, k]**2
            M1 += v2/ev[k]; M2 += v2/ev[k]**2
        self.V = M1
        self.tau = np.where(M1>1e-14, M2/M1, 0)
        self.tau_tilde = ev[k0]*self.tau
        self.degree = deg; self.N = N

TABLA = {
    'LCI': +0.267, 'BPmaps': -0.032, 'Microarray': -1.000,
    'Phenotypes': -0.999, 'IntegratedNetwork': -1.000, 'Interolog': -0.190,
    'Genetic': -0.632, 'WI8': -0.602, 'wi2004': -0.507, 'wi2007': -0.421,
}

def analizar(archivo):
    df = pd.read_csv(archivo, comment='#', header=None,
                     names=['source','target','eid','weight'])
    G = nx.Graph(); G.add_edges_from(df[['source','target']].values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    A = nx.to_numpy_array(G); s = SPG(A)
    return A.shape[0], spearmanr(s.degree,s.V)[0], spearmanr(s.degree,s.tau_tilde)[0]

print(f"{'red':20s} {'N':>5s} {'Sp(k,V)':>9s} {'Sp(k,tt)':>9s} {'tabla':>8s} {'match':>6s}")
print("-"*62)
for f in sorted(os.listdir('.')):
    if f.endswith('.csv') and f not in ('edges.csv','nodes.csv','gprops.csv'):
        red = f[:-4]
        try:
            N, rv, rtt = analizar(f)
            esp = TABLA.get(red)
            match = "OK" if (esp is not None and abs(rtt-esp)<0.15) else ("?" if esp is None else "NO")
            esp_s = f"{esp:+.3f}" if esp is not None else "nuevo"
            print(f"{red:20s} {N:5d} {rv:+.3f}   {rtt:+.3f}   {esp_s:>8s} {match:>6s}")
        except Exception as e:
            print(f"{red:20s} ERROR: {str(e)[:30]}")


red                      N   Sp(k,V)  Sp(k,tt)    tabla  match
--------------------------------------------------------------
BPmaps                 345 -0.537   +0.232     -0.032     NO
Genetic                683 -0.892   -0.317     -0.632     NO
LCI                    117 +0.071   +0.381     +0.267     OK
wi2004                1084 -0.730   -0.114     -0.507     NO


probemos otros 4

In [ ]:
from google.colab import files
subido = files.upload()
print("subiste:", list(subido.keys()))


Saving wi2007.csv to wi2007.csv
Saving Neurons.csv to Neurons.csv
Saving WI8.csv to WI8.csv
Saving TadopleLarvaBrain.csv to TadopleLarvaBrain.csv
subiste: ['wi2007.csv', 'Neurons.csv', 'WI8.csv', 'TadopleLarvaBrain.csv']


In [ ]:
import pandas as pd, networkx as nx, numpy as np, os
from scipy.stats import spearmanr

# valores de tu tabla v9 para comparar (Spearman k, tau_tilde)
TABLA = {
    'LCI': +0.267, 'BPmaps': -0.032, 'Microarray': -1.000,
    'Phenotypes': -0.999, 'IntegratedNetwork': -1.000, 'Interolog': -0.190,
    'Genetic': -0.632, 'WI8': -0.602, 'wi2004': -0.507, 'wi2007': -0.421,
}

def analizar(archivo):
    df = pd.read_csv(archivo, comment='#', header=None,
                     names=['source','target','eid','weight'])
    G = nx.Graph(); G.add_edges_from(df[['source','target']].values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    A = nx.to_numpy_array(G)
    s = SPG(A)
    rho_tt = spearmanr(s.degree, s.tau_tilde)[0]
    rho_V  = spearmanr(s.degree, s.V)[0]
    return A.shape[0], rho_V, rho_tt

# analiza todos los CSV subidos (menos los del sistema)
print(f"{'red':20s} {'N':>5s} {'Sp(k,V)':>9s} {'Sp(k,tt)':>9s} {'tabla':>8s} {'match':>6s}")
print("-"*62)
for f in sorted(os.listdir('.')):
    if f.endswith('.csv') and f not in ('edges.csv','nodes.csv','gprops.csv'):
        red = f[:-4]
        try:
            N, rv, rtt = analizar(f)
            esp = TABLA.get(red)
            match = "✓" if (esp is not None and abs(rtt-esp)<0.15) else ("?" if esp is None else "✗")
            esp_s = f"{esp:+.3f}" if esp is not None else "nuevo"
            print(f"{red:20s} {N:5d} {rv:+.3f}   {rtt:+.3f}   {esp_s:>8s} {match:>6s}")
        except Exception as e:
            print(f"{red:20s} ERROR: {str(e)[:30]}")


red                      N   Sp(k,V)  Sp(k,tt)    tabla  match
--------------------------------------------------------------
BPmaps                 345 -0.537   +0.232     -0.032      ✗
Genetic                683 -0.892   -0.317     -0.632      ✗
LCI                    117 +0.071   +0.381     +0.267      ✓
Neurons                297 -0.998   -0.947      nuevo      ?
TadopleLarvaBrain      205 -1.000   -0.988      nuevo      ?
WI8                   2214 -0.774   -0.249     -0.602      ✗
wi2004                1084 -0.730   -0.114     -0.507      ✗
wi2007                1108 -0.654   -0.044     -0.421      ✗


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/CSV'
print("¿existe la carpeta?:", os.path.isdir(CARPETA))
if os.path.isdir(CARPETA):
    print("archivos que veo:")
    for f in sorted(os.listdir(CARPETA)):
        print("  ", f)


¿existe la carpeta?: True
archivos que veo:
   BPmaps.csv
   Genetic.csv
   IntegratedNetwork.csv
   Interlog.csv
   LCI.csv
   Metabolism.csv
   Microarray.csv
   Neurons.csv
   Phenotypes.csv
   TadopleLarvaBrain.csv
   WI8.csv
   wi2004.csv
   wi2007.csv


In [ ]:
import pandas as pd, networkx as nx, numpy as np
from scipy.linalg import eigh
from scipy.stats import spearmanr

class SPG:
    def __init__(self, A):
        A = np.array(A, float); np.fill_diagonal(A, 0)
        N = A.shape[0]; deg = A.sum(1)
        ev, evec = eigh(np.diag(deg) - A)
        k0 = next((k for k in range(1, N) if ev[k] > 1e-8), None)
        if k0 is None: raise ValueError("desconectado")
        M1 = np.zeros(N); M2 = np.zeros(N)
        for k in range(k0, N):
            if ev[k] < 1e-10: continue
            v2 = evec[:, k]**2
            M1 += v2/ev[k]; M2 += v2/ev[k]**2
        self.V = M1
        self.tau = np.where(M1>1e-14, M2/M1, 0)
        self.tau_tilde = ev[k0]*self.tau
        self.degree = deg; self.N = N

def analizar(archivo):
    df = pd.read_csv(archivo, comment='#', header=None,
                     names=['source','target','eid','weight'])
    G = nx.Graph(); G.add_edges_from(df[['source','target']].values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    A = nx.to_numpy_array(G); s = SPG(A)
    return A.shape[0], spearmanr(s.degree,s.V)[0], spearmanr(s.degree,s.tau_tilde)[0]


In [ ]:
import os
from scipy.stats import spearmanr

CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/CSV'

# valores escritos en tu doc v9 (para comparar — NO son verdad absoluta)
TABLA = {
    'LCI': +0.267, 'BPmaps': -0.032, 'Microarray': -1.000,
    'Phenotypes': -0.999, 'IntegratedNetwork': -1.000, 'Interlog': -0.190,
    'Genetic': -0.632, 'WI8': -0.602, 'wi2004': -0.507, 'wi2007': -0.421,
}

print(f"{'red':20s} {'N':>6s} {'Sp(k,V)':>9s} {'Sp(k,tt)':>9s} {'doc v9':>8s} {'ΔN?':>5s}")
print("-"*64)
for f in sorted(os.listdir(CARPETA)):
    if not f.endswith('.csv'): continue
    red = f[:-4]
    try:
        N, rv, rtt = analizar(os.path.join(CARPETA, f))
        esp = TABLA.get(red)
        esp_s = f"{esp:+.3f}" if esp is not None else "nuevo"
        print(f"{red:20s} {N:6d} {rv:+.3f}   {rtt:+.3f}   {esp_s:>8s}")
    except Exception as e:
        print(f"{red:20s} ERROR: {str(e)[:35]}")


red                       N   Sp(k,V)  Sp(k,tt)   doc v9   ΔN?
----------------------------------------------------------------
BPmaps                  345 -0.537   +0.232     -0.032
Genetic                 683 -0.892   -0.317     -0.632
IntegratedNetwork      5966 -0.987   -0.823     -1.000
Interlog               2378 -0.950   -0.414     -0.190
LCI                     117 +0.071   +0.381     +0.267
Metabolism              453 -0.962   -0.844      nuevo
Microarray             2333 -0.963   +0.863     -1.000
Neurons                 297 -0.998   -0.947      nuevo
Phenotypes              889 -0.999   -0.986     -0.999
TadopleLarvaBrain       205 -1.000   -0.988      nuevo
WI8                    2214 -0.774   -0.249     -0.602
wi2004                 1084 -0.730   -0.114     -0.507
wi2007                 1108 -0.654   -0.044     -0.421


In [ ]:
import numpy as np, networkx as nx, pandas as pd
from scipy.linalg import eigh
from scipy.stats import spearmanr

def diagnostico(archivo, nombre):
    df = pd.read_csv(archivo, comment='#', header=None,
                     names=['source','target','eid','weight'])
    G = nx.Graph(); G.add_edges_from(df[['source','target']].values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    A = nx.to_numpy_array(G); N = A.shape[0]
    deg = A.sum(1)
    L = np.diag(deg) - A
    ev = eigh(L, eigvals_only=True)
    k0 = next(k for k in range(1,N) if ev[k] > 1e-8)

    print(f"\n=== {nombre} ===")
    print(f"  N = {N},  aristas = {int(A.sum()/2)},  densidad = {A.sum()/(N*(N-1)):.3f}")
    print(f"  grado: min={deg.min():.0f} max={deg.max():.0f} medio={deg.mean():.1f} CV={deg.std()/deg.mean():.2f}")
    print(f"  lambda_2 = {ev[k0]:.6e}")
    print(f"  lambda_N = {ev[-1]:.4f}")
    print(f"  gap ratio lambda_2/lambda_3 = {ev[k0]/ev[k0+1]:.4f}")
    print(f"  autovalores casi-cero (<1e-6): {(np.abs(ev)<1e-6).sum()}  <- si hay >1, red casi-disconexa")
    print(f"  rango dinamico lambda_N/lambda_2 = {ev[-1]/ev[k0]:.1f}")

CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/CSV'
import os
diagnostico(os.path.join(CARPETA,'LCI.csv'), 'LCI (+0.381)')
diagnostico(os.path.join(CARPETA,'Microarray.csv'), 'Microarray (+0.863)')



=== LCI (+0.381) ===
  N = 117,  aristas = 123,  densidad = 0.018
  grado: min=1 max=90 medio=2.1 CV=3.93
  lambda_2 = 1.131307e-02
  lambda_N = 91.0002
  gap ratio lambda_2/lambda_3 = 0.0873
  autovalores casi-cero (<1e-6): 1  <- si hay >1, red casi-disconexa
  rango dinamico lambda_N/lambda_2 = 8043.8

=== Microarray (+0.863) ===
  N = 2333,  aristas = 136859,  densidad = 0.050
  grado: min=1 max=931 medio=117.3 CV=1.44
  lambda_2 = 4.195433e-03
  lambda_N = 932.1086
  gap ratio lambda_2/lambda_3 = 0.1980
  autovalores casi-cero (<1e-6): 1  <- si hay >1, red casi-disconexa
  rango dinamico lambda_N/lambda_2 = 222172.2


In [ ]:
import os, numpy as np, networkx as nx, pandas as pd
from scipy.stats import spearmanr

CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/CSV'
filas = []
for f in sorted(os.listdir(CARPETA)):
    if not f.endswith('.csv'): continue
    try:
        df = pd.read_csv(os.path.join(CARPETA,f), comment='#', header=None,
                         names=['s','t','e','w'])
        G = nx.Graph(); G.add_edges_from(df[['s','t']].values)
        G.remove_edges_from(nx.selfloop_edges(G))
        G = G.subgraph(max(nx.connected_components(G),key=len)).copy()
        A = nx.to_numpy_array(G); s=SPG(A)
        N=A.shape[0]; deg=A.sum(1)
        dens = A.sum()/(N*(N-1))
        cv = deg.std()/deg.mean()
        rtt = spearmanr(deg, s.tau_tilde)[0]
        filas.append((f[:-4], N, dens, cv, rtt))
    except Exception as e:
        print(f"{f}: {str(e)[:30]}")

filas.sort(key=lambda x: x[2])  # ordenar por densidad
print(f"{'red':20s} {'N':>5s} {'densidad':>9s} {'CV(k)':>6s} {'Sp(k,tt)':>9s}")
print("-"*54)
for red,N,dens,cv,rtt in filas:
    signo = "  <-- POSITIVO" if rtt>0 else ""
    print(f"{red:20s} {N:5d} {dens:9.4f} {cv:6.2f} {rtt:+9.3f}{signo}")


red                      N  densidad  CV(k)  Sp(k,tt)
------------------------------------------------------
WI8                   2214    0.0014   1.87    -0.249
wi2007                1108    0.0024   1.85    -0.044
wi2004                1084    0.0027   1.85    -0.114
Interlog              2378    0.0047   1.91    -0.414
Genetic                683    0.0066   1.18    -0.317
BPmaps                 345    0.0067   1.29    +0.232  <-- POSITIVO
IntegratedNetwork     5966    0.0100   2.11    -0.823
LCI                    117    0.0181   3.93    +0.381  <-- POSITIVO
Metabolism             453    0.0198   1.87    -0.844
Neurons                297    0.0489   0.89    -0.947
Microarray            2333    0.0503   1.44    +0.863  <-- POSITIVO
Phenotypes             889    0.0574   0.93    -0.986
TadopleLarvaBrain      205    0.1231   0.62    -0.988


In [ ]:
CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/CSV'
import os
print(f"{'red':15s} {'Sp(k,V)':>9s} {'Sp(k,tt)':>9s} {'¿coinciden signo?':>18s}")
print("-"*54)
for red in ['LCI','BPmaps','Microarray','Metabolism','Neurons']:
    N, rv, rtt = analizar(os.path.join(CARPETA, red+'.csv'))
    coinc = "sí" if np.sign(rv)==np.sign(rtt) else "NO <-- sospechoso"
    print(f"{red:15s} {rv:+9.3f} {rtt:+9.3f} {coinc:>18s}")


red               Sp(k,V)  Sp(k,tt)  ¿coinciden signo?
------------------------------------------------------
LCI                +0.071    +0.381                 sí
BPmaps             -0.537    +0.232  NO <-- sospechoso
Microarray         -0.963    +0.863  NO <-- sospechoso
Metabolism         -0.962    -0.844                 sí
Neurons            -0.998    -0.947                 sí
